# exp-01 — detailed skill instructions against minimal ones

Reads the runs committed under `experiments/runs/exp-01-skill-verbosity/`,
scores every leaf, and compares the two arms.

**This notebook runs nothing.** The sessions are produced by

```bash
python experiments/exp-01-skill-verbosity/run.py --replicates 5
```

which is thousands of sessions and hours long. Keeping the run out of here is
what makes the analysis re-executable: anyone can rerun this against the
committed predictions without spending anything.

The hypothesis, the decision criteria and the findings live in
[`thinking/experiments/exp-01-skill-verbosity.md`](../../thinking/experiments/exp-01-skill-verbosity.md).
Read that first — this notebook is the arithmetic behind it, not the claim.

## The arms

Each check carries two versions of its own skill, and the harness names them
by what moved off the manifest pin:

| directory | arm | skill |
|---|---|---|
| `pinned` | **detailed** | `v1`, full stepwise instructions |
| `<check>@v2` | **minimal** | `v2`, same frontmatter and opening paragraph, nothing else |

Both are scored by the same `schema.json`, the same `eval-manifest.json` and
the same gold: the contracts sit above the version directories, so the
control is structural rather than asserted.

In [ ]:
from pathlib import Path

import pandas as pd

from soda_mmqc import cli

RUNS = Path("../../experiments/runs/exp-01-skill-verbosity").resolve()
CHECKLIST = "fig-checklist-exp01"
MODEL = "claude-sonnet-5"

assert RUNS.is_dir(), f"no runs at {RUNS}; produce them with run.py first"
sorted(p.name for p in RUNS.iterdir() if p.is_dir())

## Every leaf, scored on its own

A leaf is one `<check>/<arm>/rep-NN/` directory. `score_check` takes exactly
one, which is what keeps the evaluator general — aggregating across arms and
replicates is this notebook's job, not the harness's.

In [ ]:
def leaves(runs: Path):
    """Yield (check, arm, replicate, directory) for every scored unit."""
    for check_dir in sorted(p for p in runs.iterdir() if p.is_dir()):
        for arm_dir in sorted(p for p in check_dir.iterdir() if p.is_dir()):
            for rep_dir in sorted(p for p in arm_dir.iterdir() if p.is_dir()):
                yield check_dir.name, arm_dir.name, int(rep_dir.name.split("-")[1]), rep_dir


def arm_name(check: str, arm_dir: str) -> str:
    """`pinned` is the detailed arm; anything else moved a skill to v2."""
    return "detailed" if arm_dir == "pinned" else "minimal"


rows = []
for check, arm_dir, replicate, path in leaves(RUNS):
    result = cli.score_check(CHECKLIST, check, path, model=MODEL, save=False)
    analysis = result["agentic"]["flat"][0]["analysis"]
    for prop, summary in (analysis.get("by_property") or {}).items():
        rows.append({
            "check": check,
            "arm": arm_name(check, arm_dir),
            "replicate": replicate,
            "property": prop,
            "mean_score": summary.get("mean_score"),
        })
    row_counts = (analysis.get("by_list") or {}).get("outputs", {}).get("row_counts", {})
    rows.append({
        "check": check,
        "arm": arm_name(check, arm_dir),
        "replicate": replicate,
        "property": "__rows__",
        "correct_row": row_counts.get("correct_row"),
        "missing_row": row_counts.get("missing_row"),
        "spurious_row": row_counts.get("spurious_row"),
    })

scores = pd.DataFrame(rows)
scores.head()

## Non-response

A session that answers `outputs: []` scores as a **fully missing row set** —
`correct_row: 0`, `missing_row: N` — rather than being excluded. That is
deliberate: a skill so thin that the model returns nothing is worse, and
dropping those examples would make the failing arm look better.

Count them per arm before reading any mean, because a difference in
non-response *is* part of the result.

In [ ]:
rows_only = scores[scores["property"] == "__rows__"]
non_response = (
    rows_only.assign(empty=lambda d: d["correct_row"] == 0)
    .groupby(["check", "arm"])["empty"].sum()
    .unstack(fill_value=0)
)
non_response

## Per-check comparison

`mean_score` is the layer-2 rollup over instances that are applicable and
correctly so. Averaged over replicates, then paired by check — the arms differ
in one thing, so the comparison is within a check, never across them.

In [ ]:
per_check = (
    scores[scores["property"] != "__rows__"]
    .groupby(["check", "arm", "replicate"])["mean_score"].mean()
    .groupby(["check", "arm"]).agg(["mean", "std"])
    .unstack("arm")
)
per_check

In [ ]:
paired = per_check["mean"].assign(
    difference=lambda d: d["detailed"] - d["minimal"]
).sort_values("difference")
paired

## Read the decision criteria before the plot

The note states what each outcome would look like, committed before these
runs existed. Compare `paired["difference"]` against it — do not decide what
counts as a difference here.

Replicate spread is the yardstick: a per-check difference smaller than the
within-arm standard deviation is not a difference, however consistent its
sign.

In [ ]:
ax = paired["difference"].plot.barh(
    figsize=(7, 4),
    title="mean_score: detailed minus minimal, by check",
    xlabel="difference in mean_score",
)
ax.axvline(0, linewidth=1, color="black")
ax.figure.tight_layout()